# Preliminaries

In [1]:
# GLOBAL
import warnings

# DATA LOADING
import numpy as np
import pandas as pd
import pytz
from datetime import datetime, timedelta
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# DATA VISUALIZATION
import matplotlib.pyplot as plt

In [2]:
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None

In [3]:
def max_jobs_in_window(df, window_minutes=5):
    times = pd.to_datetime(df['Submit Time'], unit='s').sort_values().to_numpy()

    window = pd.Timedelta(minutes=window_minutes)
    max_jobs = 0
    left = 0

    for right in range(len(times)):
        while times[right] - times[left] > window:
            left += 1
        max_jobs = max(max_jobs, right - left + 1)

    return max_jobs

# F-DATA

## Data Loading

In [4]:
df = pd.read_parquet('workloads/fdata/22_04.parquet')#, engine='pyarrow')

In [5]:
df.columns

Index(['jid', 'usr', 'jnam', 'cnumr', 'cnumat', 'cnumut', 'nnumr', 'adt',
       'qdt', 'schedsdt', 'deldt', 'ec', 'elpl', 'sdt', 'edt', 'nnuma',
       'idle_time_ave', 'nnumu', 'perf1', 'perf2', 'perf3', 'perf4', 'perf5',
       'perf6', 'mszl', 'pri', 'econ', 'avgpcon', 'minpcon', 'maxpcon', 'msza',
       'mmszu', 'uctmut', 'sctmut', 'usctmut', 'jobenv_req', 'freq_req',
       'freq_alloc', 'flops', 'mbwidth', 'opint', 'pclass', 'embedding',
       'exit state', 'duration'],
      dtype='object')

In [6]:
df.shape

(424413, 45)

## Data Preparation

In [7]:
df['jid']

0         jid_8390912
1         jid_8390913
2         jid_8390914
3         jid_8390915
4         jid_8390921
             ...     
424408    jid_8760027
424409    jid_8760218
424410    jid_8760214
424411    jid_8760163
424412    jid_8760090
Name: jid, Length: 424413, dtype: object

In [8]:
df['jid'] = df['jid'].str.split('_').str[1].astype(int)

In [9]:
df["jid"].is_unique

False

In [10]:
# Number of rows that are duplicates (excluding first occurrence)
num_duplicate_rows = df.duplicated(subset="jid").sum()

print(f"Number of duplicated rows: {num_duplicate_rows}")

Number of duplicated rows: 115111


In [11]:
jid_counts = df["jid"].value_counts()

# Only jids with duplicates
duplicated_jids = jid_counts[jid_counts > 1]

print("Number of duplicated jids:", len(duplicated_jids))
print(duplicated_jids.head())

Number of duplicated jids: 18548
jid
8437064    1010
8436971    1010
8499513     172
8720873     144
8437006     105
Name: count, dtype: int64


In [12]:
OFFSET = 1_000_000

# For each jid group, count the occurrence index (0, 1, 2, ...)
dup_index = df.groupby("jid").cumcount()

# Apply offset only when dup_index > 0
df["jid"] = df["jid"] + dup_index * OFFSET

df["jid"].is_unique

True

In [13]:
df['usr'] = df['usr'].str.split('_').str[1].astype(int)

In [14]:
df['duration'] = df['duration'].replace({0: 1}).astype(int)

In [15]:
df = df[df.duration<df.elpl]

In [16]:
df.sdt = pd.to_datetime(df.sdt)
df.adt = pd.to_datetime(df.adt)

In [17]:
df['wait_time'] = ((df.sdt - df.adt).dt.total_seconds()).astype(int)

In [18]:
init_ts = int(df.adt.min().timestamp())
initial_time = datetime.utcfromtimestamp(init_ts).replace(tzinfo=pytz.UTC)
df['submit_time_sec'] = ((df.adt - initial_time).dt.total_seconds()).astype(int)

In [19]:
df['group'] = 0
df['gnumr'] = -1

In [20]:
df.nnuma.max()

np.int64(82944)

In [21]:
df.elpl = df.elpl.astype(int)
df.mmszu = df.mmszu.astype(int)
# df.mszl = df.mszl.astype(int) # Too big

In [22]:
min(df.mmszu)

23003136

In [23]:
required_columns = ['jid', 'submit_time_sec', 'wait_time', 'duration', 'nnuma', 'duration', 'mmszu', 'nnumr', 'elpl', 'mmszu', 'ec', 'usr', 'group']

In [24]:
swf = df[required_columns]

In [25]:
column_names = [
    "Job Number", "Submit Time", "Wait Time", "Run Time", "Number of Allocated Nodes", "Average CPU Time Used",
    "Used Memory", "Requested Number of Nodes", "Requested Time", "Requested Memory", "Status", "User ID",
    "Group ID"]

swf.columns = column_names

In [26]:
swf['Executable Number'] = -1
swf['Queue Number'] = -1
swf['Partition Number'] = -1
swf['Preceding Job Number'] = -1
swf['Think Time from Preceding Job'] = -1

## Data Splitting

In [27]:
# Sort the dataframe by 'Submit Time'
swf = swf.sort_values(by='Submit Time', ascending=True)

# Calculate split index for 70-30 split
split_index = int(len(df) * 0.8)

# Split the dataframe
swf_train = swf.iloc[:split_index]
swf_test = swf.iloc[split_index:]

In [28]:
swf_train.shape

(339504, 18)

In [29]:
swf_test.shape

(84877, 18)

In [30]:
max_jobs = max_jobs_in_window(swf_test, window_minutes=5)
print(max_jobs)

1760


In [32]:
def analyze_dataset(swf_df, train_df, test_df, dataset_name, original_shape):
    print(f"=== {dataset_name} STATS ===")

    # 1. Total, Train, Test shapes
    total_jobs = len(swf_df)
    train_jobs = len(train_df)
    test_jobs = len(test_df)
    dropped_jobs = original_shape[0] - total_jobs

    print(f"Original Row Count: {original_shape[0]}")
    print(f"Dropped Jobs during cleaning: {dropped_jobs}")
    print(f"Total Cleaned Jobs: {total_jobs}")
    print(f"Train Jobs: {train_jobs} ({train_jobs/total_jobs*100:.1f}%)")
    print(f"Test Jobs: {test_jobs} ({test_jobs/total_jobs*100:.1f}%)")

    # 2. Unique Users
    unique_users = swf_df["User ID"].nunique()
    print(f"Unique Users: {unique_users}")

    # 3. Repeated / Duplicate Jobs (Same user, resources, and runtime)

    duplicate_groups = swf_df.groupby(
        [
            "User ID",
            "Requested Number of Nodes",
            "Requested Time",
            "Run Time",
        ]
    )

    total_duplicates = 0
    for name, group in duplicate_groups:
        if len(group) > 1:
            # The first job is the original, subsequent ones are duplicates
            total_duplicates += len(group) - 1

    dup_percentage = (total_duplicates / total_jobs) * 100
    print(f"Repeated/Duplicate Jobs: {total_duplicates} ({dup_percentage:.2f}%)")
    print("-" * 30)

analyze_dataset(swf, swf_train, swf_test, "F-DATA", (424413, 45))

=== F-DATA STATS ===
Original Row Count: 424413
Dropped Jobs during cleaning: 32
Total Cleaned Jobs: 424381
Train Jobs: 339504 (80.0%)
Test Jobs: 84877 (20.0%)
Unique Users: 491
Repeated/Duplicate Jobs: 324987 (76.58%)
------------------------------


## Final Data Preparation

In [30]:
initial_time = int(swf_test['Submit Time'].min())

# Calculate the difference in seconds:
swf_test['Submit Time'] = swf_test['Submit Time'] - initial_time

In [31]:
# The maximum number of submitted jobs in a five minutes window is roughly 1800. 
# The system has 159k nodes, so it can handle this rate easily.
# To stress the system, we will consider a 1000x decrease in job resources.

In [32]:
swf_test_filtered = swf_test[swf_test["Number of Allocated Nodes"]<15897]
print(f"There are {len(swf_test_filtered)} out of {len(swf_test)} with less than 158976 nodes allocated")

There are 84870 out of 84877 with less than 158976 nodes allocated


## File Creation

In [31]:
swf_test.to_csv('workloads/fdata/fdata.swf', sep='\t', index=False, header=False)

In [33]:
swf_test["Requested Memory"].describe()

count    8.487700e+04
mean     2.652770e+08
std      8.257673e+08
min     -2.147484e+09
25%      2.335703e+08
50%      4.314235e+08
75%      5.656412e+08
max      2.147156e+09
Name: Requested Memory, dtype: float64

In [34]:
min(swf_test["Requested Memory"])

-2147483648

In [35]:
max(swf_test["Requested Memory"])

2147155968